# Satarkta Model Training for Kaggle (GPU) with MLG-ULB Credit Card Dataset
This notebook trains the XGBoost model utilizing Kaggle's GPU capabilities on the classic highly-imbalanced credit card fraud dataset.

In [1]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, accuracy_score
import os

# Load the dataset from Kaggle inputs
dataset_path = '/kaggle/input/datasets/organizations/mlg-ulb/creditcardfraud/creditcard.csv'
print("Loading Credit Card Fraud dataset...")
df = pd.read_csv(dataset_path)
print(f"Loaded {len(df)} rows. Fraud cases: {df['Class'].sum()}")

# Select Features and Target
features = ['Time', 'Amount'] + [f'V{i}' for i in range(1, 29)]
X = df[features]
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Training XGBoost Classifier on GPU...")

# Kaggle GPU configuration
clf = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    tree_method='hist',
    device='cuda', # Enables GPU acceleration
    scale_pos_weight=99, # Handle extreme class imbalance
    objective='binary:logistic',
    random_state=42
)

clf.fit(X_train, y_train)

# Predictions
y_pred = clf.predict(X_test)
y_scores = clf.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
auprc = average_precision_score(y_test, y_scores)

print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test AUPRC (Average Precision): {auprc:.4f}")

# Save Model Artifact
clf.save_model('model.json')
print("\nModel saved as 'model.json'.")


Loading Credit Card Fraud dataset...
Loaded 284807 rows. Fraud cases: 492
Training XGBoost Classifier on GPU...
Test Accuracy: 0.9995
Test AUPRC (Average Precision): 0.8819

Model saved as 'model.json'.


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [15:21:49] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)
